<a href="https://colab.research.google.com/github/Mohsin7979/Data-Scraping-From-Website-Zameen.com/blob/main/Data_Scraping_From_Website_Zameen_com.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step 2 — Install required libraries**

In [ ]:
!pip install requests beautifulsoup4 pandas lxml

**Step 3 — Import libraries**

Create a new cell:

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

**Step 4 — Request the Zameen page**

Create another cell:

In [ ]:
url = "https://www.zameen.com/agents/"

response = requests.get(
    url,
    headers={
        "User-Agent": "Mozilla/5.0"
    },
    timeout=20
)

print("Status Code:", response.status_code)

Status Code: 200


**Step 5 — Parse the HTML**

Now:

In [ ]:
soup = BeautifulSoup(response.text, "lxml")

print(soup.title.get_text(strip=True))

Top Real Estate Agents in Pakistan - Zameen.com


**Step 6 — See all links**
Before extracting data, it's very useful to understand the page.

In [ ]:
links = soup.find_all("a")

print("Total links:", len(links))

for link in links[:30]:
    print(
        link.get_text(" ", strip=True),
        "=>",
        link.get("href")
    )

Total links: 109
 => https://www.zameen.com
Properties => https://www.zameen.com/
PROPERTY BLOCKS => https://www.myzameen.com/
Area Guides => https://www.zameen.com/area-guides/
Blog => https://www.zameen.com/blog/
Maps => https://www.zameen.com/society_maps/
Plot Finder => https://www.zameen.com/plotfinder/
Home Loan Calculator => /tools/home-loan-calculator/
Area Unit Converter => https://www.zameen.com/tools/area-unit-converter
Land Record Pages => https://www.zameen.com/tools/land-records
Construction Cost Calculator => https://www.zameen.com/tools/construction-cost-calculator
Forum => https://www.zameen.com/forum/
Index => https://www.zameen.com/index/
Trends => https://www.zameen.com/trends.html
Add Property => https://www.zameen.com/add_property_single.html
 => https://www.zameen.com/ur
 => https://www.zameen.com
Homes => https://www.zameen.com/
Plots => https://www.zameen.com/plots.html
Commercial => https://www.zameen.com/commercial.html
rent => https://www.zameen.com/rentals.

**Step 7 — Find agent/agency links**
The current Zameen agents pages contain links to individual agency profiles.

In [ ]:
agent_links = []

for link in soup.find_all("a", href=True):

    href = link["href"]
    text = link.get_text(" ", strip=True)

    if "/agents/" in href and text:
        agent_links.append({
            "name": text,
            "url": href
        })

print("Found:", len(agent_links))

for item in agent_links[:10]:
    print(item)

Found: 34
{'name': 'Karachi 1473 Agencies View Trend', 'url': '/agents/Karachi-2/?page=1'}
{'name': 'Lahore 1527 Agencies View Trend', 'url': '/agents/Lahore-1/?page=1'}
{'name': 'Islamabad 1431 Agencies View Trend', 'url': '/agents/Islamabad-3/?page=1'}
{'name': 'Gwadar 5 Agencies View Trend', 'url': '/agents/Gwadar-389/?page=1'}
{'name': 'Faisalabad 96 Agencies View Trend', 'url': '/agents/Faisalabad-16/?page=1'}
{'name': 'Gujranwala 89 Agencies View Trend', 'url': '/agents/Gujranwala-327/?page=1'}
{'name': 'Peshawar 94 Agencies View Trend', 'url': '/agents/Peshawar-17/?page=1'}
{'name': 'Multan 117 Agencies View Trend', 'url': '/agents/Multan-15/?page=1'}
{'name': 'Rawalpindi 336 Agencies View Trend', 'url': '/agents/Rawalpindi-41/?page=1'}
{'name': 'Sialkot 33 Agencies View Trend', 'url': '/agents/Sialkot-480/?page=1'}


**Step 8 — Remove duplicate links**
(Webpages often contain repeated links)

In [ ]:
unique_agents = {}

for item in agent_links:
    unique_agents[item["url"]] = item["name"]

print("Unique agents:", len(unique_agents))

Unique agents: 34


Now convert them back to a list:

In [ ]:
agents = []

for url, name in unique_agents.items():

    agents.append({
        "agency_name": name,
        "profile_url": url
    })

print(agents[:5])

[{'agency_name': 'Karachi 1473 Agencies View Trend', 'profile_url': '/agents/Karachi-2/?page=1'}, {'agency_name': 'Lahore 1527 Agencies View Trend', 'profile_url': '/agents/Lahore-1/?page=1'}, {'agency_name': 'Islamabad 1431 Agencies View Trend', 'profile_url': '/agents/Islamabad-3/?page=1'}, {'agency_name': 'Gwadar 5 Agencies View Trend', 'profile_url': '/agents/Gwadar-389/?page=1'}, {'agency_name': 'Faisalabad 96 Agencies View Trend', 'profile_url': '/agents/Faisalabad-16/?page=1'}]


**Step 9 — Create a Pandas DataFrame**

In [ ]:
df = pd.DataFrame(agents)

df.head(10)

,agency_name,profile_url
0,Karachi 1473 Agencies View Trend,/agents/Karachi-2/?page=1
1,Lahore 1527 Agencies View Trend,/agents/Lahore-1/?page=1
2,Islamabad 1431 Agencies View Trend,/agents/Islamabad-3/?page=1
3,Gwadar 5 Agencies View Trend,/agents/Gwadar-389/?page=1
4,Faisalabad 96 Agencies View Trend,/agents/Faisalabad-16/?page=1
5,Gujranwala 89 Agencies View Trend,/agents/Gujranwala-327/?page=1
6,Peshawar 94 Agencies View Trend,/agents/Peshawar-17/?page=1
7,Multan 117 Agencies View Trend,/agents/Multan-15/?page=1
8,Rawalpindi 336 Agencies View Trend,/agents/Rawalpindi-41/?page=1
9,Sialkot 33 Agencies View Trend,/agents/Sialkot-480/?page=1


**Step 10 — Make profile URLs complete**

Some links may be relative, such as:

/agents/Lahore/Lahore_Agents-175696/

Convert them into complete URLs:

In [ ]:
from urllib.parse import urljoin

df["profile_url"] = df["profile_url"].apply(
    lambda x: urljoin("https://www.zameen.com", x)
)

df.head()

,agency_name,profile_url
0,Karachi 1473 Agencies View Trend,https://www.zameen.com/agents/Karachi-2/?page=1
1,Lahore 1527 Agencies View Trend,https://www.zameen.com/agents/Lahore-1/?page=1
2,Islamabad 1431 Agencies View Trend,https://www.zameen.com/agents/Islamabad-3/?page=1
3,Gwadar 5 Agencies View Trend,https://www.zameen.com/agents/Gwadar-389/?page=1
4,Faisalabad 96 Agencies View Trend,https://www.zameen.com/agents/Faisalabad-16/?p...


**Step 11 — Save your first dataset**

In [ ]:
df.to_csv(
    "zameen_agents_links.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV saved successfully!")

CSV saved successfully!


**Step 12 — Download CSV from Colab**

In [ ]:
from google.colab import files

files.download("zameen_agents_links.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Step 13 — Scrape individual agency profiles

This is where the project becomes more interesting.

For example, Zameen has individual agency pages that can contain information such as:

Agency name
City
Agency staff
Description

The public profile example I checked contains Lahore, staff information, and an agency description.

We can visit each profile URL one at a time.

In [ ]:
def get_page(url):

    response = requests.get(
        url,
        headers={
            "User-Agent": "Mozilla/5.0"
        },
        timeout=20
    )

    if response.status_code == 200:
        return BeautifulSoup(response.text, "lxml")

    print("Failed:", response.status_code, url)
    return None

**Step 14 — Test one agency**

Take the first profile:

In [ ]:
first_url = df.iloc[0]["profile_url"]

print(first_url)

https://www.zameen.com/agents/Karachi-2/?page=1


**Step 15 — Extract page text**

For learning purposes, let's first see the text:
if profile_soup:

    text = profile_soup.get_text(
        " ",
        strip=True
    )

    print(text[:3000])
   This is important because it lets you understand what information is actually available before creating selectors.

**Step 16 — Extract the page heading**

In [ ]:
display(df.head(5))

,agency_name,profile_url
0,Karachi 1473 Agencies View Trend,https://www.zameen.com/agents/Karachi-2/?page=1
1,Lahore 1527 Agencies View Trend,https://www.zameen.com/agents/Lahore-1/?page=1
2,Islamabad 1431 Agencies View Trend,https://www.zameen.com/agents/Islamabad-3/?page=1
3,Gwadar 5 Agencies View Trend,https://www.zameen.com/agents/Gwadar-389/?page=1
4,Faisalabad 96 Agencies View Trend,https://www.zameen.com/agents/Faisalabad-16/?p...


**Step 17 — Create a function for agency details**

You can now make a reusable function:

In [ ]:
def scrape_agency(url):

    soup = get_page(url)

    if soup is None:
        return None

    data = {
        "profile_url": url,
        "agency_name": "",
        "page_title": ""
    }

    if soup.title:
        data["page_title"] = soup.title.get_text(
            " ",
            strip=True
        )

    h1 = soup.find("h1")

    if h1:
        data["agency_name"] = h1.get_text(
            " ",
            strip=True
        )

    return data

**Step 18 — Test the function**

In [ ]:
result = scrape_agency(first_url)

print(result)

{'profile_url': 'https://www.zameen.com/agents/Karachi-2/?page=1', 'agency_name': '1,472 Property Agents in Karachi', 'page_title': 'Top Real Estate Agents in Karachi - Zameen.com'}


**Step 19 — Scrape multiple agencies**

For testing, start with only 5 profiles.

In [ ]:
results = []

for url in df["profile_url"].head(5):

    print("Scraping:", url)

    result = scrape_agency(url)

    if result:
        results.append(result)

    time.sleep(2)

Scraping: https://www.zameen.com/agents/Karachi-2/?page=1
Scraping: https://www.zameen.com/agents/Lahore-1/?page=1
Scraping: https://www.zameen.com/agents/Islamabad-3/?page=1
Scraping: https://www.zameen.com/agents/Gwadar-389/?page=1
Scraping: https://www.zameen.com/agents/Faisalabad-16/?page=1


**Step 20 — Convert results to DataFrame**

In [ ]:
details_df = pd.DataFrame(results)

details_df

,profile_url,agency_name,page_title
0,https://www.zameen.com/agents/Karachi-2/?page=1,"1,472 Property Agents in Karachi",Top Real Estate Agents in Karachi - Zameen.com
1,https://www.zameen.com/agents/Lahore-1/?page=1,"1,525 Property Agents in Lahore",Top Real Estate Agents in Lahore - Zameen.com
2,https://www.zameen.com/agents/Islamabad-3/?page=1,"1,428 Property Agents in Islamabad",Top Real Estate Agents in Islamabad - Zameen.com
3,https://www.zameen.com/agents/Gwadar-389/?page=1,5 Property Agents in Gwadar,Top Real Estate Agents in Gwadar - Zameen.com
4,https://www.zameen.com/agents/Faisalabad-16/?p...,96 Property Agents in Faisalabad,Top Real Estate Agents in Faisalabad - Zameen.com


**Step 21 — Save the detailed data**

In [ ]:
details_df.to_csv(
    "zameen_agents_details.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully!")

Saved successfully!


**Download:**

In [ ]:
files.download("zameen_agents_details.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Step 22 — Scrape multiple listing pages**

The agents section currently has separate listing pages such as Agents-1, Agents-2, etc.

For practice, you can test 3 pages:

In [ ]:
pages = [
    "https://www.zameen.com/agents/",
    "https://www.zameen.com/agents/Agents-1/",
    "https://www.zameen.com/agents/Agents-2/"
]

Then:

In [ ]:
all_agents = []

for page_url in pages:

    print("Opening:", page_url)

    response = requests.get(
        page_url,
        headers={
            "User-Agent": "Mozilla/5.0"
        },
        timeout=20
    )

    if response.status_code != 200:
        print("Failed:", response.status_code)
        continue

    soup = BeautifulSoup(response.text, "lxml")

    for link in soup.find_all("a", href=True):

        href = link["href"]
        name = link.get_text(" ", strip=True)

        if "/agents/" in href and name:

            full_url = urljoin(
                "https://www.zameen.com",
                href
            )

            all_agents.append({
                "agency_name": name,
                "profile_url": full_url
            })

    time.sleep(3)

Opening: https://www.zameen.com/agents/
Opening: https://www.zameen.com/agents/Agents-1/
Opening: https://www.zameen.com/agents/Agents-2/


**Step 23 — Remove duplicates**

In [ ]:
all_df = pd.DataFrame(all_agents)

all_df = all_df.drop_duplicates(
    subset=["profile_url"]
)

all_df = all_df.reset_index(drop=True)

print("Total unique profiles:", len(all_df))

all_df.head()

Total unique profiles: 55


,agency_name,profile_url
0,Karachi 1473 Agencies View Trend,https://www.zameen.com/agents/Karachi-2/?page=1
1,Lahore 1527 Agencies View Trend,https://www.zameen.com/agents/Lahore-1/?page=1
2,Islamabad 1431 Agencies View Trend,https://www.zameen.com/agents/Islamabad-3/?page=1
3,Gwadar 5 Agencies View Trend,https://www.zameen.com/agents/Gwadar-389/?page=1
4,Faisalabad 96 Agencies View Trend,https://www.zameen.com/agents/Faisalabad-16/?p...


**Step 24 — Save final CSV**

In [ ]:
all_df.to_csv(
    "zameen_all_agents.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Final CSV created!")

Final CSV created!


In [ ]:
files.download("zameen_all_agents.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>